In [1]:
import pandas as pd
import numpy as np
import re

In [2]:
ruta = "RESULTADOS-YUCATÁN.xlsx"

# 1. Cargar hojas
df_sitios = pd.read_excel(ruta, sheet_name="SITIOS")
df_resultados = pd.read_excel(ruta, sheet_name="RESULTADOS")
df_simbologia = pd.read_excel(ruta, sheet_name="SIMBOLOGÍA")

# 2. Limpiar texto en SITIOS (quitar espacios extra y puntos al final)
cols_texto_sitios = df_sitios.select_dtypes(include="object").columns
for col in cols_texto_sitios:
    df_sitios[col] = (
        df_sitios[col]
        .astype(str)
        .str.strip()
        .str.rstrip(".")
        .replace({"nan": np.nan, "None": np.nan})
    )

# 3. Convertir fecha en RESULTADOS
df_resultados["FECHA REALIZACIÓN"] = pd.to_datetime(
    df_resultados["FECHA REALIZACIÓN"], errors="coerce"
)

# 4. Función para limpiar valores con '<' o '>' en parámetros numéricos
def limpiar_medicion(val, factor_menor=0.5):
    if pd.isna(val):
        return np.nan
    if isinstance(val, (int, float)):
        return float(val)
    
    val_str = str(val).strip().replace(",", "")
    if val_str.startswith("<"):
        num = re.sub(r"[^\d.]", "", val_str)
        return float(num) * factor_menor if num else np.nan
    elif val_str.startswith(">"):
        num = re.sub(r"[^\d.]", "", val_str)
        return float(num) if num else np.nan
    try:
        return float(val_str)
    except ValueError:
        return np.nan

# Identificar columnas de metadatos vs columnas de parámetros
cols_meta = [
    "CLAVE SITIO", "CLAVE DE MONITOREO", "NOMBRE DEL SITIO",
    "TIPO CUERPO DE AGUA", "FECHA REALIZACIÓN", "Año"
]
cols_parametros = [c for c in df_resultados.columns if c not in cols_meta]

# Aplicar limpieza numérica a todos los parámetros
for col in cols_parametros:
    df_resultados[col] = df_resultados[col].apply(limpiar_medicion)

# 5. Unir coordenadas y municipio desde SITIOS
df_limpio = df_resultados.merge(
    df_sitios[["CLAVE SITIO", "MUNICIPIO", "ACUÍFERO", "SUBTIPO CUERPO AGUA", "LATITUD", "LONGITUD"]],
    on="CLAVE SITIO",
    how="left"
)

print(f"Dataset integrado: {df_limpio.shape[0]} filas x {df_limpio.shape[1]} columnas")

C:\Users\zanch\AppData\Local\Temp\ipykernel_22440\278892082.py:9: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cols_texto_sitios = df_sitios.select_dtypes(include="object").columns


Dataset integrado: 1323 filas x 219 columnas


In [7]:
df_limpio.dtypes

CLAVE SITIO                       str
CLAVE DE MONITOREO                str
NOMBRE DEL SITIO                  str
TIPO CUERPO DE AGUA               str
FECHA REALIZACIÓN      datetime64[us]
                            ...      
MUNICIPIO                         str
ACUÍFERO                          str
SUBTIPO CUERPO AGUA               str
LATITUD                       float64
LONGITUD                      float64
Length: 219, dtype: object

In [12]:
# 1. Definir las columnas a conservar
columnas_seleccionadas = [
    # Ubicación y tiempo
    "NOMBRE DEL SITIO",
    "FECHA REALIZACIÓN",
    "Año",
    "MUNICIPIO",
    "LONGITUD",
    "LATITUD",
    # Compuestos nitrogenados
    "N_NO3", "N_NO2", "N_NO3-N_NO2",
    # Metales pesados y metaloides
    "AS_TOT", "AS_SOL",
    "PB_TOT", "PB_SOL",
    "HG_TOT", "HG_SOL",
    "CD_TOT", "CD_SOL",
    "MN_TOT", "MN_SOL",
    # Patógenos microbiológicos
    "COLI_FEC", "ENTEROC_FEC", "E_COLI", "H_HELMINTOS",
    # Subproductos de desinfección (DBPs)
    "THMs", "CLOROFORMO", "BR_DICLMET", "DIBR_CLMET", "BROMOF",
    "ACID_CLOROACE", "ACID_DICLOROACE", "ACID_TRICLOROACE",
    # Compuestos Orgánicos Volátiles (COVs) y otros
    "BENCENO", "TRICLOROETILENO", "TETRACLOROETILENO_",
    "TETRACL_CARB", "CLORURO_DE_VINILO",
    "FLUORUROS_TOT", "CN_TOT"
]

# 2. Crear el nuevo DataFrame filtrado
df_contaminantes_infantil = df_limpio[columnas_seleccionadas].copy()

# 3. Exportar a CSV (utf-8-sig conserva acentos y la "ñ" al abrirlo en Excel)
nombre_archivo = "contaminantes_yucatan_salud_infantil.csv"
df_contaminantes_infantil.to_csv(nombre_archivo, index=False, encoding="utf-8-sig")

print(f"Archivo '{nombre_archivo}' guardado con {df_contaminantes_infantil.shape[0]} filas y {df_contaminantes_infantil.shape[1]} columnas.")
df_contaminantes_infantil.head()

Archivo 'contaminantes_yucatan_salud_infantil.csv' guardado con 1323 filas y 38 columnas.


,NOMBRE DEL SITIO,FECHA REALIZACIÓN,Año,MUNICIPIO,LONGITUD,LATITUD,N_NO3,N_NO2,N_NO3-N_NO2,AS_TOT,...,ACID_CLOROACE,ACID_DICLOROACE,ACID_TRICLOROACE,BENCENO,TRICLOROETILENO,TETRACLOROETILENO_,TETRACL_CARB,CLORURO_DE_VINILO,FLUORUROS_TOT,CN_TOT
0,POZO 4 DE LA PLANTA MERIDA IV,2012-12-20,2012,UMÁN,-89.784621,20.971536,4.170000,0.002500,NaN,0.00075,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,POZO 4 DE LA PLANTA MERIDA IV,2013-05-03,2013,UMÁN,-89.784621,20.971536,4.134700,0.002500,NaN,0.00075,...,NaN,NaN,NaN,0.00007,NaN,0.0001,0.000016,0.000075,0.2686,NaN
2,POZO 4 DE LA PLANTA MERIDA IV,2017-10-25,2017,UMÁN,-89.784621,20.971536,3.905300,0.005220,NaN,0.00075,...,NaN,NaN,NaN,0.00007,0.0001,0.0001,0.000016,0.000075,0.1457,NaN
3,POZO 4 DE LA PLANTA MERIDA IV,2018-05-02,2018,UMÁN,-89.784621,20.971536,2.145212,0.002500,NaN,0.00075,...,NaN,NaN,NaN,0.00007,0.0001,0.0001,0.000016,0.000075,0.1913,NaN
4,POZO 4 DE LA PLANTA MERIDA IV,2018-12-20,2018,UMÁN,-89.784621,20.971536,3.476133,0.007668,NaN,0.00075,...,NaN,NaN,NaN,0.00007,0.0001,0.0001,0.000016,0.000075,0.2277,NaN


In [14]:

import unicodedata

# 1. Cargar hojas desde el Excel original
ruta = "RESULTADOS-YUCATÁN.xlsx"
df_sitios = pd.read_excel(ruta, sheet_name="SITIOS")
df_res = pd.read_excel(ruta, sheet_name="RESULTADOS")

# 2. Normalizar texto (quitar acentos, mayúsculas, espacios y puntos)
def quitar_acentos(texto):
    if pd.isna(texto):
        return texto
    return ''.join(
        c for c in unicodedata.normalize('NFD', str(texto))
        if unicodedata.category(c) != 'Mn'
    ).upper().strip().rstrip('.')

df_sitios["MUNICIPIO"] = df_sitios["MUNICIPIO"].apply(quitar_acentos)
df_sitios["NOMBRE DEL SITIO"] = df_sitios["NOMBRE DEL SITIO"].apply(quitar_acentos)

# 3. Limpiar mediciones con '<' o '>'
def limpiar_medicion(val, factor_menor=0.5):
    if pd.isna(val):
        return np.nan
    if isinstance(val, (int, float)):
        return float(val)
    val_str = str(val).strip().replace(",", "")
    if val_str.startswith("<"):
        num = re.sub(r"[^\d.]", "", val_str)
        return float(num) * factor_menor if num else np.nan
    elif val_str.startswith(">"):
        num = re.sub(r"[^\d.]", "", val_str)
        return float(num) if num else np.nan
    try:
        return float(val_str)
    except ValueError:
        return np.nan

columnas_seleccionadas = [
    "NOMBRE DEL SITIO", "FECHA REALIZACIÓN", "Año", "MUNICIPIO", "LONGITUD", "LATITUD",
    "N_NO3", "N_NO2", "N_NO3-N_NO2",
    "AS_TOT", "AS_SOL", "PB_TOT", "PB_SOL", "HG_TOT", "HG_SOL", "CD_TOT", "CD_SOL", "MN_TOT", "MN_SOL",
    "COLI_FEC", "ENTEROC_FEC", "E_COLI", "H_HELMINTOS",
    "THMs", "CLOROFORMO", "BR_DICLMET", "DIBR_CLMET", "BROMOF",
    "ACID_CLOROACE", "ACID_DICLOROACE", "ACID_TRICLOROACE",
    "BENCENO", "TRICLOROETILENO", "TETRACLOROETILENO_", "TETRACL_CARB", "CLORURO_DE_VINILO",
    "FLUORUROS_TOT", "CN_TOT"
]

cols_contaminantes = [
    c for c in columnas_seleccionadas
    if c not in ["NOMBRE DEL SITIO", "FECHA REALIZACIÓN", "Año", "MUNICIPIO", "LONGITUD", "LATITUD"]
]

for col in cols_contaminantes:
    if col in df_res.columns:
        df_res[col] = df_res[col].apply(limpiar_medicion)

df_res["FECHA REALIZACIÓN"] = pd.to_datetime(df_res["FECHA REALIZACIÓN"], errors="coerce")

# 4. Unir con SITIOS ya normalizados
df_limpio = df_res.merge(
    df_sitios[["CLAVE SITIO", "MUNICIPIO", "LATITUD", "LONGITUD"]],
    on="CLAVE SITIO",
    how="left"
)[columnas_seleccionadas].copy()

# 5. Identificar municipios con registros elevados (umbrales NOM-127-SSA1-2021 / microbiológicos)
umbrales_elevados = {
    "N_NO3": 11.0,        # mg/L
    "N_NO2": 0.9,         # mg/L
    "AS_TOT": 0.01,       # mg/L
    "PB_TOT": 0.01,       # mg/L
    "HG_TOT": 0.001,      # mg/L
    "CD_TOT": 0.003,      # mg/L
    "MN_TOT": 0.15,       # mg/L
    "FLUORUROS_TOT": 1.5, # mg/L
    "COLI_FEC": 1000.0    # NMP/100 mL
}

municipios_elevados = set()
for col, limite in umbrales_elevados.items():
    muns = df_limpio[df_limpio[col] > limite]["MUNICIPIO"].dropna().unique()
    municipios_elevados.update(muns)

# 6. Filtrar manteniendo TODOS los años de esos municipios y ordenar por MUNICIPIO y Año
df_municipios_elevados = (
    df_limpio[df_limpio["MUNICIPIO"].isin(municipios_elevados)]
    .sort_values(by=["MUNICIPIO", "Año", "FECHA REALIZACIÓN"])
    .reset_index(drop=True)
)

# 7. Exportar a CSV (todas las entradas individuales agrupadas/ordenadas por municipio y año)
archivo_salida = "municipios_elevados_historico_2012_2025.csv"
df_municipios_elevados.to_csv(archivo_salida, index=False, encoding="utf-8-sig")

print(f"Municipios seleccionados ({len(municipios_elevados)}): {sorted(municipios_elevados)}")
print(f"CSV guardado: '{archivo_salida}' ({df_municipios_elevados.shape[0]} filas x {df_municipios_elevados.shape[1]} columnas)")

# Vista previa de MERIDA ordenada por año
df_municipios_elevados.head(10)

Municipios seleccionados (29): ['CELESTUN', 'CHAPAB', 'CHEMAX', 'CHOCHOLA', 'CUZAMA', 'DZILAM DE BRAVO', 'ESPITA', 'HALACHO', 'HOMUN', 'HUNUCMA', 'KOPOMA', 'MERIDA', 'MOTUL', 'MUNA', 'PROGRESO', 'RIO LAGARTOS', 'SAN FELIPE', 'TEKAX', 'TELCHAC PUERTO', 'TEMOZON', 'TEPAKAN', 'TICUL', 'TIMUCUY', 'TIXPEHUAL', 'TIZIMIN', 'UMAN', 'VALLADOLID', 'YAXKUKUL', 'YOBAIN']
CSV guardado: 'municipios_elevados_historico_2012_2025.csv' (1179 filas x 38 columnas)


,NOMBRE DEL SITIO,FECHA REALIZACIÓN,Año,MUNICIPIO,LONGITUD,LATITUD,N_NO3,N_NO2,N_NO3-N_NO2,AS_TOT,...,ACID_CLOROACE,ACID_DICLOROACE,ACID_TRICLOROACE,BENCENO,TRICLOROETILENO,TETRACLOROETILENO_,TETRACL_CARB,CLORURO_DE_VINILO,FLUORUROS_TOT,CN_TOT
0,RESERVA DE LA BIOSFERA RIA CELESTUN 3,2012-11-25,2012,CELESTUN,-90.349740,20.933200,0.4898,0.0025,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,RESERVA DE LA BIOSFERA RIA CELESTUN 1,2012-11-25,2012,CELESTUN,-90.352820,20.911120,0.4854,0.0025,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,RESERVA DE LA BIOSFERA,2012-11-25,2012,CELESTUN,-90.354485,20.888298,0.2011,0.0025,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,RESERVA DE LA BIOSFERA CELESTUN,2012-11-25,2012,CELESTUN,-90.381840,20.857900,0.3958,0.0025,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,RESERVA DE LA BIOSFERA,2012-12-25,2012,CELESTUN,-90.354485,20.888298,0.6306,0.0902,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,RESERVA DE LA BIOSFERA RIA CELESTUN 2,2012-12-25,2012,CELESTUN,-90.381500,20.846830,2.5258,0.0025,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,RESERVA DE LA BIOSFERA CELESTUN,2012-12-25,2012,CELESTUN,-90.381840,20.857900,0.6904,0.1130,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,POZO 1 DEL SISTEMA DE AGUA POTABLE DE CELESTUN,2012-12-25,2012,CELESTUN,-90.235560,20.850340,2.5087,0.0025,NaN,0.00075,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0182,NaN
8,RESERVA DE LA BIOSFERA RIA CELESTUN 3,2013-03-20,2013,CELESTUN,-90.349740,20.933200,0.6311,0.0296,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,RESERVA DE LA BIOSFERA RIA CELESTUN 1,2013-03-20,2013,CELESTUN,-90.352820,20.911120,0.6507,0.0025,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [16]:


# Ruta exacta a tu archivo local en Windows
ruta_entrada = r"C:\Users\zanch\Programas\3er cuatrimestre\Concurso de data\fetales_base_de_datos_2012_csv\conjunto_de_datos\fetales_2012_unificado.csv"

# Catálogo oficial de entidades (INEGI)
catalogo_entidades = {
    1: "AGUASCALIENTES", 2: "BAJA CALIFORNIA", 3: "BAJA CALIFORNIA SUR", 4: "CAMPECHE",
    5: "COAHUILA DE ZARAGOZA", 6: "COLIMA", 7: "CHIAPAS", 8: "CHIHUAHUA",
    9: "CIUDAD DE MEXICO", 10: "DURANGO", 11: "GUANAJUATO", 12: "GUERRERO",
    13: "HIDALGO", 14: "JALISCO", 15: "MEXICO", 16: "MICHOACAN DE OCAMPO",
    17: "MORELOS", 18: "NAYARIT", 19: "NUEVO LEON", 20: "OAXACA",
    21: "PUEBLA", 22: "QUERETARO", 23: "QUINTANA ROO", 24: "SAN LUIS POTOSI",
    25: "SINALOA", 26: "SONORA", 27: "TABASCO", 28: "TAMAULIPAS",
    29: "TLAXCALA", 30: "VERACRUZ DE IGNACIO DE LA LLAVE", 31: "YUCATAN", 32: "ZACATECAS",
    33: "ESTADOS UNIDOS DE NORTEAMERICA", 34: "OTROS PAISES DE LATINOAMERICA",
    35: "OTROS PAISES", 99: "NO ESPECIFICADO"
}

# Catálogo oficial de los 106 municipios de Yucatán (Clave INEGI 31)
catalogo_municipios_yucatan = {
    1: "ABALA", 2: "ACANCEH", 3: "AKIL", 4: "BACA", 5: "BOKOBA",
    6: "BUCTZOTZ", 7: "CACALCHEN", 8: "CALOTMUL", 9: "CANSAHCAB", 10: "CANTAMAYEC",
    11: "CELESTUN", 12: "CENOTILLO", 13: "CONKAL", 14: "CUNCUNUL", 15: "CUZAMA",
    16: "CHACSINKIN", 17: "CHANKOM", 18: "CHAPAB", 19: "CHEMAX", 20: "CHICXULUB PUEBLO",
    21: "CHICHIMILA", 22: "CHIKINDZONOT", 23: "CHOCHOLA", 24: "CHUMAYEL", 25: "DZAN",
    26: "DZEMUL", 27: "DZIDZANTUN", 28: "DZILAM DE BRAVO", 29: "DZILAM GONZALEZ", 30: "DZITAS",
    31: "DZONCAUICH", 32: "ESPITA", 33: "HALACHO", 34: "HOCABA", 35: "HOCTUN",
    36: "HOMUN", 37: "HUHI", 38: "HUNUCMA", 39: "IXIL", 40: "IZAMAL",
    41: "KANASIN", 42: "KANTUNIL", 43: "KAUA", 44: "KINCHIL", 45: "KOPOMA",
    46: "MAMA", 47: "MANI", 48: "MAXCANU", 49: "MAYAPAN", 50: "MERIDA",
    51: "MOCOCHA", 52: "MOTUL", 53: "MUNA", 54: "MUXUPIP", 55: "OPICHEN",
    56: "OXKUTZCAB", 57: "PANABA", 58: "PETO", 59: "PROGRESO", 60: "QUINTANA ROO",
    61: "RIO LAGARTOS", 62: "SACALUM", 63: "SAMAHIL", 64: "SANAHCAT", 65: "SAN FELIPE",
    66: "SANTA ELENA", 67: "SEYE", 68: "SINANCHE", 69: "SOTUTA", 70: "SUCILA",
    71: "SUDZAL", 72: "SUMA", 73: "TAHDZIU", 74: "TAHMEK", 75: "TEABO",
    76: "TECOH", 77: "TEKAL DE VENEGAS", 78: "TEKANTO", 79: "TEKAX", 80: "TEKIT",
    81: "TEKOM", 82: "TELCHAC PUEBLO", 83: "TELCHAC PUERTO", 84: "TEMAX", 85: "TEMOZON",
    86: "TEPAKAN", 87: "TETIZ", 88: "TEYA", 89: "TICUL", 90: "TIMUCUY",
    91: "TINUM", 92: "TIXCACALCUPUL", 93: "TIXKOKOB", 94: "TIXMEHUAC", 95: "TIXPEHUAL",
    96: "TIZIMIN", 97: "TUNKAS", 98: "TZUCACAB", 99: "UAYMA", 100: "UCU",
    101: "UMAN", 102: "VALLADOLID", 103: "XOCCHEL", 104: "YAXCABA", 105: "YAXKUKUL",
    106: "YOBAIN", 999: "NO ESPECIFICADO"
}

def limpiar_fetales_yucatan(ruta_csv):
    df = pd.read_csv(ruta_csv)
    
    # 1. Filtrar solo residencia habitual en Yucatán (clave 31)
    df_yuc = df[df["ent_resid"] == 31].copy()
    
    # 2. Agregar nombres de entidades
    df_yuc["ent_regis_desc"] = df_yuc["ent_regis"].map(catalogo_entidades)
    df_yuc["ent_resid_desc"] = df_yuc["ent_resid"].map(catalogo_entidades)
    df_yuc["ent_ocurr_desc"] = df_yuc["ent_ocurr"].map(catalogo_entidades)
    
    # 3. Agregar nombres de municipios de Yucatán
    df_yuc["mun_regis_desc"] = df_yuc.apply(
        lambda r: catalogo_municipios_yucatan.get(r["mun_regis"], "OTRO ESTADO") if r["ent_regis"] == 31 else "OTRO ESTADO",
        axis=1
    )
    df_yuc["mun_resid_desc"] = df_yuc["mun_resid"].map(catalogo_municipios_yucatan)
    df_yuc["mun_ocurr_desc"] = df_yuc.apply(
        lambda r: catalogo_municipios_yucatan.get(r["mun_ocurr"], "OTRO ESTADO") if r["ent_ocurr"] == 31 else "OTRO ESTADO",
        axis=1
    )
    
    return df_yuc

# Ejecutar y guardar en la misma carpeta de trabajo del notebook
df_fetales_2012 = limpiar_fetales_yucatan(ruta_entrada)
df_fetales_2012.to_csv("fetales_2012_yucatan_limpio.csv", index=False, encoding="utf-8-sig")

print(f"Listo: {df_fetales_2012.shape[0]} filas y {df_fetales_2012.shape[1]} columnas guardadas.")
df_fetales_2012[["ent_resid_desc", "mun_resid_desc", "causa_def_desc", "pes_prod", "eda_prod"]].head()

Listo: 317 filas y 101 columnas guardadas.


,ent_resid_desc,mun_resid_desc,causa_def_desc,pes_prod,eda_prod
4489,YUCATAN,MERIDA,Otras afecciones especificadas originadas en e...,405,21
18614,YUCATAN,NO ESPECIFICADO,Feto y recién nacido afectados por otras compl...,550,22
18637,YUCATAN,PROGRESO,Hipoxia intrauterina notada por primera vez an...,3170,40
22093,YUCATAN,PROGRESO,Feto y recién nacido afectados por otra compre...,530,23
22094,YUCATAN,TEKAX,Otras afecciones especificadas originadas en e...,2600,37


In [17]:
# Lista de los 29 municipios con lecturas altas de contaminantes tóxicos
municipios_elevados = [
    "CELESTUN", "CHAPAB", "CHEMAX", "CHOCHOLA", "CUZAMA", "DZILAM DE BRAVO",
    "ESPITA", "HALACHO", "HOMUN", "HUNUCMA", "KOPOMA", "MERIDA", "MOTUL",
    "MUNA", "PROGRESO", "RIO LAGARTOS", "SAN FELIPE", "TEKAX",
    "TELCHAC PUERTO", "TEMOZON", "TEPAKAN", "TICUL", "TIMUCUY", "TIXPEHUAL",
    "TIZIMIN", "UMAN", "VALLADOLID", "YAXKUKUL", "YOBAIN"
]

# Opción alternativa: tomarlos directamente del CSV de agua que generaste antes
# municipios_elevados = pd.read_csv("municipios_elevados_historico_2012_2025.csv")["MUNICIPIO"].unique()

# Filtrar por municipio de residencia habitual de la madre
df_fetales_2012_toxicos = df_fetales_2012[
    df_fetales_2012["mun_resid_desc"].isin(municipios_elevados)
].copy()

# Guardar resultado en CSV
df_fetales_2012_toxicos.to_csv(
    "fetales_2012_yucatan_municipios_toxicos.csv",
    index=False,
    encoding="utf-8-sig"
)

print(f"Filtrado listo: {df_fetales_2012_toxicos.shape[0]} filas x {df_fetales_2012_toxicos.shape[1]} columnas")
df_fetales_2012_toxicos["mun_resid_desc"].value_counts()

Filtrado listo: 213 filas x 101 columnas


mun_resid_desc
MERIDA            109
TIZIMIN            19
CHEMAX             15
VALLADOLID         15
PROGRESO            7
TEKAX               7
UMAN                7
MOTUL               6
ESPITA              4
TICUL               3
CHAPAB              3
HALACHO             2
MUNA                2
CELESTUN            2
TEMOZON             2
YOBAIN              2
TIMUCUY             2
HUNUCMA             2
CHOCHOLA            1
TEPAKAN             1
HOMUN               1
TELCHAC PUERTO      1
Name: count, dtype: int64

In [18]:

# Cargar el dataset nacional limpio/unificado y el estatal de Yucatán
df_nac = pd.read_csv(r"C:\Users\zanch\Programas\3er cuatrimestre\Concurso de data\fetales_base_de_datos_2012_csv\conjunto_de_datos\fetales_2012_unificado.csv")
df_yuc = pd.read_csv("fetales_2012_yucatan_limpio.csv")

# 1. Calcular promedios nacionales (entidades 1..32, municipio especificado != 999)
df_nac_valido = df_nac[(df_nac["ent_resid"].between(1, 32)) & (df_nac["mun_resid"] != 999)]
conteo_nac = df_nac_valido.groupby(["ent_resid", "mun_resid"]).size()

prom_nac_con_reg = conteo_nac.mean()           # 13.63
prom_nac_total = len(df_nac_valido) / 2457     # 9.25 (sobre 2,457 municipios en México)

# 2. Calcular promedios estatales de Yucatán (clave 31, municipio != 999)
df_yuc_valido = df_yuc[df_yuc["mun_resid"] != 999]
conteo_yuc = df_yuc_valido.groupby("mun_resid_desc").size()

prom_yuc_con_reg = conteo_yuc.mean()           # 5.25 (sobre 60 municipios con al menos 1 caso)
prom_yuc_106 = len(df_yuc_valido) / 106        # 2.97 (sobre los 106 municipios de Yucatán)

# 3. Evaluar los 29 municipios con contaminantes elevados
municipios_elevados = sorted([
    "CELESTUN", "CHAPAB", "CHEMAX", "CHOCHOLA", "CUZAMA", "DZILAM DE BRAVO",
    "ESPITA", "HALACHO", "HOMUN", "HUNUCMA", "KOPOMA", "MERIDA", "MOTUL",
    "MUNA", "PROGRESO", "RIO LAGARTOS", "SAN FELIPE", "TEKAX",
    "TELCHAC PUERTO", "TEMOZON", "TEPAKAN", "TICUL", "TIMUCUY", "TIXPEHUAL",
    "TIZIMIN", "UMAN", "VALLADOLID", "YAXKUKUL", "YOBAIN"
])

comparacion = []
for mun in municipios_elevados:
    muertes = int(conteo_yuc.get(mun, 0))
    comparacion.append({
        "Municipio": mun,
        "Muertes_2012": muertes,
        "Sobre_Prom_Yuc_106": muertes > prom_yuc_106,
        "Sobre_Prom_Yuc_ConReg": muertes > prom_yuc_con_reg,
        "Sobre_Prom_Nac_Total": muertes > prom_nac_total,
        "Sobre_Prom_Nac_ConReg": muertes > prom_nac_con_reg
    })

df_comparacion = pd.DataFrame(comparacion).sort_values(by=["Muertes_2012", "Municipio"], ascending=[False, True])
display(df_comparacion)

,Municipio,Muertes_2012,Sobre_Prom_Yuc_106,Sobre_Prom_Yuc_ConReg,Sobre_Prom_Nac_Total,Sobre_Prom_Nac_ConReg
11,MERIDA,109,True,True,True,True
24,TIZIMIN,19,True,True,True,True
2,CHEMAX,15,True,True,True,True
26,VALLADOLID,15,True,True,True,True
14,PROGRESO,7,True,True,False,False
17,TEKAX,7,True,True,False,False
25,UMAN,7,True,True,False,False
12,MOTUL,6,True,True,False,False
6,ESPITA,4,True,False,False,False
1,CHAPAB,3,True,False,False,False


In [19]:

# 1. Cargar el CSV de nacimientos y defunciones 2025
df = pd.read_csv("nacimientos_y_defunciones_fetales_yucatan_2025.csv")

# 2. Normalizar nombres de municipios para cruzar sin problemas de acentos
def quitar_acentos(texto):
    if pd.isna(texto):
        return texto
    return ''.join(
        c for c in unicodedata.normalize('NFD', str(texto))
        if unicodedata.category(c) != 'Mn'
    ).upper().strip()

df["municipio_norm"] = df["municipio"].apply(quitar_acentos)

# 3. Calcular porcentaje (fallecidos / nacimientos * 100) y promedios de referencia
df["porcentaje_raw"] = (df["defunciones_fetales_2025"] / df["nacimientos_2025"]) * 100
promedio_estatal = df["porcentaje_raw"].mean()  # 1.7746%

# 4. Filtrar los 29 municipios marcados como peligrosos
municipios_peligrosos = [
    "CELESTUN", "CHAPAB", "CHEMAX", "CHOCHOLA", "CUZAMA", "DZILAM DE BRAVO",
    "ESPITA", "HALACHO", "HOMUN", "HUNUCMA", "KOPOMA", "MERIDA", "MOTUL",
    "MUNA", "PROGRESO", "RIO LAGARTOS", "SAN FELIPE", "TEKAX",
    "TELCHAC PUERTO", "TEMOZON", "TEPAKAN", "TICUL", "TIMUCUY", "TIXPEHUAL",
    "TIZIMIN", "UMAN", "VALLADOLID", "YAXKUKUL", "YOBAIN"
]

df_peligrosos = df[df["municipio_norm"].isin(municipios_peligrosos)].copy()
promedio_grupo = df_peligrosos["porcentaje_raw"].mean()  # 1.7436%

# 5. Agregar columnas finales y exportar a CSV
df_peligrosos["porcentaje_fallecidos"] = df_peligrosos["porcentaje_raw"].round(2)
df_peligrosos["arriba_promedio_estatal"] = df_peligrosos["porcentaje_raw"].apply(
    lambda x: "Sí" if x > promedio_estatal else "No"
)
df_peligrosos["arriba_promedio_grupo"] = df_peligrosos["porcentaje_raw"].apply(
    lambda x: "Sí" if x > promedio_grupo else "No"
)

df_final = (
    df_peligrosos
    .drop(columns=["municipio_norm", "porcentaje_raw"])
    .sort_values(by="porcentaje_fallecidos", ascending=False)
    .reset_index(drop=True)
)

df_final.to_csv("municipios_peligrosos_fetales_vs_nacimientos_2025.csv", index=False, encoding="utf-8-sig")
display(df_final)

,cve_entidad,entidad,cve_municipio,municipio,nacimientos_2025,defunciones_fetales_2025,porcentaje_fallecidos,arriba_promedio_estatal,arriba_promedio_grupo
0,31,Yucatán,106,Yobaín,15,1,6.67,Sí,Sí
1,31,Yucatán,45,Kopomá,23,1,4.35,Sí,Sí
2,31,Yucatán,61,Río Lagartos,25,1,4.00,Sí,Sí
3,31,Yucatán,53,Muna,102,4,3.92,Sí,Sí
4,31,Yucatán,36,Homún,57,2,3.51,Sí,Sí
5,31,Yucatán,33,Halachó,236,8,3.39,Sí,Sí
6,31,Yucatán,11,Celestún,64,2,3.12,Sí,Sí
7,31,Yucatán,52,Motul,322,9,2.80,Sí,Sí
8,31,Yucatán,15,Cuzamá,41,1,2.44,Sí,Sí
9,31,Yucatán,23,Chocholá,43,1,2.33,Sí,Sí
